In [48]:
import pickle
from tensorflow.keras.models import load_model

kaggle_model = load_model("kaggle_text_model.keras")
trap_model = load_model("trap4phish_model.keras")
spam_model = load_model("spamassassin_model.keras")

with open("kaggle_tokenizer.pkl", "rb") as f:
    kaggle_tokenizer = pickle.load(f)

with open("spamassassin_tokenizer.pkl", "rb") as f:
    spam_tokenizer = pickle.load(f)

with open("trap4phish_scaler.pkl", "rb") as f:
    trap_scaler = pickle.load(f)

print("All models and preprocessing files loaded successfully.")

/home/moaz/projects/Email forensics/.venv/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:801: UserWarning: Skipping variable loading for optimizer 'adam', because it has 14 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
/home/moaz/projects/Email forensics/.venv/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:801: UserWarning: Skipping variable loading for optimizer 'adam', because it has 24 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


All models and preprocessing files loaded successfully.


In [49]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

In [50]:
def predict_kaggle_email(email_text):
    seq = kaggle_tokenizer.texts_to_sequences([email_text])
    padded = pad_sequences(seq, maxlen=200)
    prob = float(kaggle_model.predict(padded)[0][0])
    if prob >= 0.5:
        label = "Phishing"
    else:
        label = "Legitimate"
    return {
        "model": "Kaggle Phishing Model",
        "prediction": label,
        "confidence_score": prob
    }


In [51]:

sample_email = "Your account has been suspended. Click this link immediately to verify your password."
predict_kaggle_email(sample_email)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 486ms/step


{'model': 'Kaggle Phishing Model',
 'prediction': 'Phishing',
 'confidence_score': 0.5061964988708496}

In [52]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

In [53]:
def predict_spamassassin_email(email_text):
    seq = spam_tokenizer.texts_to_sequences([email_text])
    padded = pad_sequences(seq, maxlen=200)
    prob = float(spam_model.predict(padded)[0][0])
    if prob >= 0.5:
        label = "Spam/Phishing"
    else:
        label = "Legitimate"
    return {
        "model": "SpamAssassin Model",
        "prediction": label,
        "confidence_score": prob
    }

In [54]:
predict_spamassassin_email(sample_email)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 492ms/step


{'model': 'SpamAssassin Model',
 'prediction': 'Legitimate',
 'confidence_score': 0.49994370341300964}

## Trap4Phish Integration

Trap4Phish uses spreadsheet/macro-based numerical features, so a real Trap4Phish feature vector is used for integration testing. Raw SpamAssassin emails do not contain these spreadsheet feature values directly.

In [55]:
import pandas as pd

def predict_trap4phish(features):

    features_df = pd.DataFrame(
        [features],
        columns=trap_scaler.feature_names_in_
    )

    features_scaled = trap_scaler.transform(features_df)

    prob = float(trap_model.predict(features_scaled)[0][0])

    if prob >= 0.5:
        label = "Phishing"
    else:
        label = "Legitimate"

    return {
        "model": "Trap4Phish Model",
        "prediction": label,
        "confidence_score": prob
    }


In [33]:
sample_features = [0.2] * 10
predict_trap4phish(sample_features)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step


{'model': 'Trap4Phish Model',
 'prediction': 'Legitimate',
 'confidence_score': 0.49498695135116577}

In [100]:
def analyze_email_text(email_text, trap_features=None):
    kaggle_result = predict_kaggle_email(email_text)
    spam_result = predict_spamassassin_email(email_text)

    results = {
        "email_text": email_text,
        "kaggle_model_result": kaggle_result,
        "spamassassin_model_result": spam_result
    }

    if trap_features is not None:
        trap_result = predict_trap4phish(trap_features)
        results["trap4phish_model_result"] = trap_result
    else:
        trap_result = None

    phishing_votes = 0
    legitimate_votes = 0
    borderline_models = []

    # Kaggle vote
    if kaggle_result["prediction"] == "Phishing":
        phishing_votes += 1
    else:
        legitimate_votes += 1

    if 0.45 <= kaggle_result["confidence_score"] <= 0.55:
        borderline_models.append("Kaggle")

    # SpamAssassin vote
    if spam_result["prediction"] == "Spam/Phishing":
        phishing_votes += 1
    else:
        legitimate_votes += 1

    if 0.45 <= spam_result["confidence_score"] <= 0.55:
        borderline_models.append("SpamAssassin")

    # Trap4Phish vote
    if trap_result is not None:
        if trap_result["prediction"] == "Phishing":
            phishing_votes += 1
        else:
            legitimate_votes += 1

        if 0.45 <= trap_result["confidence_score"] <= 0.55:
            borderline_models.append("Trap4Phish")

    # Majority vote final verdict
    if phishing_votes > legitimate_votes:
        final_verdict = "Suspicious / Potential Phishing"
    elif legitimate_votes > phishing_votes:
        final_verdict = "Likely Legitimate"
    else:
        final_verdict = "Requires Manual Review"

    results["phishing_votes"] = phishing_votes
    results["legitimate_votes"] = legitimate_votes
    results["borderline_models"] = borderline_models
    results["final_verdict"] = final_verdict

    return results

In [101]:
analyze_email_text(sample_email)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step


{'email_text': 'Your account has been suspended. Click this link immediately to verify your password.',
 'kaggle_model_result': {'model': 'Kaggle Phishing Model',
  'prediction': 'Phishing',
  'confidence_score': 0.5061964988708496},
 'spamassassin_model_result': {'model': 'SpamAssassin Model',
  'prediction': 'Legitimate',
  'confidence_score': 0.49994370341300964},
 'phishing_votes': 1,
 'legitimate_votes': 1,
 'borderline_models': ['Kaggle', 'SpamAssassin'],
 'final_verdict': 'Requires Manual Review'}

In [57]:
def generate_forensic_report(email_text):
    analysis = analyze_email_text(email_text)
    print("====== EMAIL FORENSIC REPORT ======")
    print()
    print("Email Content:")
    print(email_text)
    print()
    print("Kaggle Model Prediction:")
    print(analysis["kaggle_model_result"])
    print()
    print("SpamAssassin Model Prediction:")
    print(analysis["spamassassin_model_result"])
    print()
    print("Final Verdict:")
    print(analysis["final_verdict"])

In [37]:
generate_forensic_report(sample_email)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
====== EMAIL FORENSIC REPORT ======

Email Content:
Your account has been suspended. Click this link immediately to verify your password.

Kaggle Model Prediction:
{'model': 'Kaggle Phishing Model', 'prediction': 'Phishing', 'confidence_score': 0.5061964988708496}

SpamAssassin Model Prediction:
{'model': 'SpamAssassin Model', 'prediction': 'Legitimate', 'confidence_score': 0.49994370341300964}

Final Verdict:
Suspicious / Potential Phishing


In [58]:
def generate_full_report(email_text, trap_features=None):
    text_analysis = analyze_email_text(email_text)
    print("====== EMAIL FORENSIC REPORT ======")
    print("Email Content:")
    print(email_text)
    print()
    print("Kaggle Model Prediction:")
    print(text_analysis["kaggle_model_result"])
    print()
    print("SpamAssassin Model Prediction:")
    print(text_analysis["spamassassin_model_result"])
    print()
    if trap_features is not None:
        trap_result = predict_trap4phish(trap_features)
        print("Trap4Phish Model Prediction:")
        print(trap_result)
        print()
    print("Final Verdict:")
    print(text_analysis["final_verdict"])

In [88]:
generate_full_report(sample_email, sample_features)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
====== EMAIL FORENSIC REPORT ======
Email Content:
Your account has been suspended. Click this link immediately to verify your password.

Kaggle Model Prediction:
{'model': 'Kaggle Phishing Model', 'prediction': 'Phishing', 'confidence_score': 0.5061964988708496}

SpamAssassin Model Prediction:
{'model': 'SpamAssassin Model', 'prediction': 'Legitimate', 'confidence_score': 0.49994370341300964}

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Trap4Phish Model Prediction:
{'model': 'Trap4Phish Model', 'prediction': 'Legitimate', 'confidence_score': 0.49498695135116577}

Final Verdict:
Requires Manual Review


In [60]:
from email import message_from_string
from email.policy import default

def parse_email(raw_email):
    msg = message_from_string(raw_email, policy=default)

    parsed_data = {
        "from": msg.get("From"),
        "to": msg.get("To"),
        "subject": msg.get("Subject"),
        "date": msg.get("Date"),
        "return_path": msg.get("Return-Path"),
        "received_headers": msg.get_all("Received", []),
        "authentication_results": msg.get("Authentication-Results")
    }

    return parsed_data

In [43]:
email_path = "Datasets/spamassassin/20021010_spam/spam/0498.863566df8e5f17f979edca79d1e87187"

with open(email_path, "r", encoding="latin-1") as f:
    raw_email = f.read()

parse_email(raw_email)

{'from': 'Answer.Us@davicom.co.kr',
 'to': 'undisclosed-recipients:;',
 'subject': '$10 a hour for watching e-mmercials! No joke!',
 'date': 'Mon, 07 Oct 2002 22:57:45 -0400',
 'return_path': '<bounce2@u-answer.com>',
 'received_headers': ['from localhost (jalapeno [127.0.0.1])\tby zzzzason.org (Postfix) with ESMTP id 10C9616F18\tfor <zzzz@localhost>; Tue,  8 Oct 2002 11:02:05 +0100 (IST)',
  'from jalapeno [127.0.0.1]\tby localhost with IMAP (fetchmail-5.9.0)\tfor zzzz@localhost (single-drop); Tue, 08 Oct 2002 11:02:05 +0100 (IST)',
  'from webnote.net (mail.webnote.net [193.120.211.219]) by    dogma.slashnull.org (8.11.6/8.11.6) with ESMTP id g98339K30569 for    <zzzz@jmason.org>; Tue, 8 Oct 2002 04:03:10 +0100',
  'from davicom.co.kr (IDENT:root@[211.205.76.5]) by webnote.net    (8.9.3/8.9.3) with ESMTP id EAA21173 for <zzzz@example.com>;    Tue, 8 Oct 2002 04:03:55 +0100',
  'from u-answer.com (kenny.answer-us.com [12.43.213.7]) by    davicom.co.kr (v3smtp 8.11.2 Copyright (c) 1998

In [61]:
def analyze_headers(parsed_email):
    findings = []
    if parsed_email["return_path"]:
        findings.append("Return-Path found")
    if parsed_email["received_headers"]:
        findings.append(f"{len(parsed_email['received_headers'])} Received headers detected")
    if parsed_email["authentication_results"] is None:
        findings.append("No SPF/DKIM/DMARC authentication results present")
    return findings


In [45]:
parsed_email = parse_email(raw_email)
analyze_headers(parsed_email)


['Return-Path found',
 '5 Received headers detected',
 'No SPF/DKIM/DMARC authentication results present']

In [102]:
def generate_integrated_report(raw_email, trap_features=None):
    parsed_email = parse_email(raw_email)
    header_findings = analyze_headers(parsed_email)

    email_text = clean_email_body(raw_email)
    ml_analysis = analyze_email_text(email_text, trap_features)

    print("====== INTEGRATED EMAIL FORENSIC REPORT ======\n")

    print("--- Parsed Email Details ---")
    print("From:", parsed_email.get("from"))
    print("To:", parsed_email.get("to"))
    print("Subject:", parsed_email.get("subject"))
    print("Date:", parsed_email.get("date"))
    print("Return-Path:", parsed_email.get("return_path"))

    print("\n--- Header Analysis Findings ---")
    for finding in header_findings:
        print("-", finding)

    print("\n--- ML Detection Results ---")
    kaggle = ml_analysis["kaggle_model_result"]
    spamassassin = ml_analysis["spamassassin_model_result"]

    print("Kaggle Model:", kaggle["prediction"], "| Score:", round(kaggle["confidence_score"], 4))
    print("SpamAssassin Model:", spamassassin["prediction"], "| Score:", round(spamassassin["confidence_score"], 4))

    if "trap4phish_model_result" in ml_analysis:
        trap = ml_analysis["trap4phish_model_result"]
        print("Trap4Phish Model:", trap["prediction"], "| Score:", round(trap["confidence_score"], 4))

    print("\n--- Decision Summary ---")
    print("Phishing Votes:", ml_analysis["phishing_votes"])
    print("Legitimate Votes:", ml_analysis["legitimate_votes"])
    print("Borderline Models:", ml_analysis["borderline_models"])

    print("\n--- Final Verdict ---")
    print(ml_analysis["final_verdict"])

    return ml_analysis

In [103]:
report = generate_integrated_report(raw_email, sample_features)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
====== INTEGRATED EMAIL FORENSIC REPORT ======

--- Parsed Email Details ---
From: Answer.Us@davicom.co.kr
To: undisclosed-recipients:;
Subject: $10 a hour for watching e-mmercials! No joke!
Date: Mon, 07 Oct 2002 22:57:45 -0400
Return-Path: <bounce2@u-answer.com>

--- Header Analysis Findings ---
- Return-Path found
- 5 Received headers detected
- No SPF/DKIM/DMARC authentication results present

--- ML Detection Results ---
Kaggle Model: Phishing | Score: 0.5054
SpamAssassin Model: Legitimate | Score: 0.4967
Trap4Phish Model: Legitimate | Score: 0.495

--- Decision Summary ---
Phishing Votes: 1
Legitimate Votes: 2
Borderline Models: ['Kaggle', 'SpamAssassin', 'Trap4Phish']

--- Final Verdict ---
Likely Legitimate


In [72]:
from email import policy
from email.parser import Parser

def extract_email_body(raw_email):
    msg = Parser(policy=policy.default).parsestr(raw_email)

    if msg.is_multipart():
        for part in msg.walk():
            content_type = part.get_content_type()

            if content_type == "text/plain":
                try:
                    return part.get_content()
                except:
                    payload = part.get_payload(decode=True)
                    return payload.decode("latin-1", errors="ignore")

            if content_type == "text/html":
                try:
                    return part.get_content()
                except:
                    payload = part.get_payload(decode=True)
                    return payload.decode("latin-1", errors="ignore")

    else:
        try:
            return msg.get_content()
        except:
            payload = msg.get_payload(decode=True)
            if payload:
                return payload.decode("latin-1", errors="ignore")
            return str(msg.get_payload())

    return ""

In [ ]:
email_body = extract_email_body(raw_email)
print(email_body[:1000])

PROFESSIONAL, EFFECTIVE DEBT COLLECTION SERVICES AVAILABLE

For the last seventeen years, National Credit Systems, Inc. has been providing
top flight debt collection services to over 15,000 businesses, institutions, and 
healthcare providers.

We charge only a low-flat fee (less than $20) per account, and all proceeds are 
forwarded to you directly -- not to your collections agency.

If you wish, we will report unpaid accounts to Experian (formerly TRW), 
TRANSUNION, and Equifax. There is no charge for this important service.

PLEASE LET US KNOW IF WE CAN BE OF SERVICE TO YOU.

Simply reply to debt_collectors@chmailnet.com with the following instructions 
in the Subject field - 

REMOVE  --  Please remove me from your mailing list.
EMAIL   --  Please email more information.
FAX     --  Please fax more information.
MAIL    --  Please snailmail more information.
CALL    --  Please have a representative call.

Indicate the best time to telephone and any necessary addresses and 
telephone/

In [ ]:
import subprocess
subprocess.check_call(["pip", "install", "beautifulsoup4"])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 1.0 MB/s eta 0:00:00a 0:00:01m


0

In [73]:
from bs4 import BeautifulSoup
def clean_email_body(raw_email):
    body = extract_email_body(raw_email)
    soup = BeautifulSoup(body, "html.parser")
    clean_text = soup.get_text(separator=" ", strip=True)
    return clean_text

In [74]:
clean_body = clean_email_body(raw_email)
print(clean_body[:1000])

Answer-Us Unlist Information This message is brought to you by Answer-us.com in 
		compliance with current federal laws. To find out more about Answer-us.com visit http://www.answer-us.com . 
        You are receiving this mailing because you or someone you know has 
        registered this email address to receive special offers from an Answer-us.com 
        marketing partner. Screening of addresses has been done to the best of 
        our knowledge. We honor all unlist requests within 72 hours. If you
        have received this email in error, we apologize for any inconvenience it 
        has caused and will not mail further offers to you. To be
        unlisted from our database, please do the 
        following: Simply click here . If you have your mail forwarded to a new email address 
        please provide your old email address. A nswer - Us Nationwide support@answer-us.com Our 
        E-Mail Campaigns Have Produced
      Staggering Response Rates! Responsive 
      General

In [ ]:
print(clean_email_body(raw_email)[:300])

PROFESSIONAL, EFFECTIVE DEBT COLLECTION SERVICES AVAILABLE

For the last seventeen years, National Credit Systems, Inc. has been providing
top flight debt collection services to over 15,000 businesses, institutions, and 
healthcare providers.

We charge only a low-flat fee (less than $20) per accoun


## Integrated Email Forensic Analysis Pipeline

This module integrates email parsing, header analysis, machine learning-based phishing detection, and forensic verdict generation into a single investigation workflow.

In [ ]:
def generate_integrated_report(raw_email, trap_features=None):
    parsed_email = parse_email(raw_email)
    header_findings = analyze_headers(parsed_email)
    email_text = clean_email_body
    print("====== INTEGRATED EMAIL FORENSIC REPORT ======")
    print("\n--- Parsed Email Details ---")
    print("From:", parsed_email["from"])
    print("To:", parsed_email["to"])
    print("Subject:", parsed_email["subject"])
    print("Date:", parsed_email["date"])
    print("Return-Path:", parsed_email["return_path"])
    print("\n--- Header Analysis Findings ---")
    for finding in header_findings:
        print("-", finding)
    print("\n--- ML Detection Results ---")
    print("Kaggle:", ml_analysis["kaggle_model_result"]["prediction"],
          "| Score:", ml_analysis["kaggle_model_result"]["confidence_score"])
    print("SpamAssassin:", ml_analysis["spamassassin_model_result"]["prediction"],
          "| Score:", ml_analysis["spamassassin_model_result"]["confidence_score"])
    if "trap4phish_model_result" in ml_analysis:
        print("Trap4Phish:", ml_analysis["trap4phish_model_result"]["prediction"],
              "| Score:", ml_analysis["trap4phish_model_result"]["confidence_score"])
    print("\n--- Final Verdict ---")
    print(ml_analysis["final_verdict"])

In [ ]:
generate_integrated_report(raw_email, sample_features)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
====== INTEGRATED EMAIL FORENSIC REPORT ======

--- Parsed Email Details ---
From: Answer.Us@davicom.co.kr
To: undisclosed-recipients:;
Subject: $10 a hour for watching e-mmercials! No joke!
Date: Mon, 07 Oct 2002 22:57:45 -0400
Return-Path: <bounce2@u-answer.com>

--- Header Analysis Findings ---
- Return-Path found
- 5 Received headers detected
- No SPF/DKIM/DMARC authentication results present

--- ML Detection Results ---
Kaggle: Phishing | Score: 0.5053941607475281
SpamAssassin: Legitimate | Score: 0.49671536684036255
Trap4Phish: Legitimate | Score: 0.49498695135116577

--- Final Verdict ---
Suspicious / Potential Phishing


In [ ]:
import pandas as pd

trap_df = pd.read_csv("Datasets/Trap4phish/Trap4phish.csv")
trap_df.head()

,file_name,entropy_of_text,macro_chr_count,macro_vocab_size,macro_arithmetic_operator_count,macro_token_count,macro_max_line_length,remote_template_present,numeric_cell_count,string_cell_count,avg_cell_length,label
0,benign_sample_3253.xlsx,5.746068,39,10079,4759,10079,38,1,4336,8926,11.184588,0
1,benign_sample_5685.xlsx,5.749956,287,19789,28783,19789,42,1,27138,54355,11.167818,0
2,benign_sample_8732.xlsx,5.745470,122,16084,12754,16084,38,1,12033,23997,11.132195,0
3,benign_sample_5743.xlsx,5.743675,152,17382,16244,17382,39,1,15217,30554,11.104673,0
4,benign_sample_5522.xlsx,5.748130,106,15452,11529,15452,38,1,10837,21622,11.151976,0


In [ ]:
trap_df.columns

Index(['file_name', 'entropy_of_text', 'macro_chr_count', 'macro_vocab_size',
       'macro_arithmetic_operator_count', 'macro_token_count',
       'macro_max_line_length', 'remote_template_present',
       'numeric_cell_count', 'string_cell_count', 'avg_cell_length', 'label'],
      dtype='str')

In [ ]:
feature_cols = [
    "entropy_of_text",
    "macro_chr_count",
    "macro_vocab_size",
    "macro_arithmetic_operator_count",
    "macro_token_count",
    "macro_max_line_length",
    "remote_template_present",
    "numeric_cell_count",
    "string_cell_count",
    "avg_cell_length"
]

sample_features = trap_df.loc[0, feature_cols].tolist()

sample_features

[5.746067900571417,
 39.0,
 10079.0,
 4759.0,
 10079.0,
 38.0,
 1.0,
 4336.0,
 8926.0,
 11.18458754335696]

In [ ]:
type(trap_scaler)

sklearn.preprocessing._data.StandardScaler

In [ ]:
trap_scaler.feature_names_in_

array(['entropy_of_text', 'macro_chr_count', 'macro_vocab_size',
       'macro_arithmetic_operator_count', 'macro_token_count',
       'macro_max_line_length', 'remote_template_present',
       'numeric_cell_count', 'string_cell_count', 'avg_cell_length'],
      dtype=object)

In [ ]:
import os

spam_folder = "Datasets/spamassassin/20021010_spam/spam"

print("Total spam emails:", len(os.listdir(spam_folder)))

Total spam emails: 501


In [ ]:
spam_files = os.listdir(spam_folder)[:5]

for file in spam_files:
    print("\n==============================")
    print("Testing file:", file)

    email_path = os.path.join(spam_folder, file)

    with open(email_path, "r", encoding="latin-1") as f:
        raw_email = f.read()

    generate_integrated_report(raw_email, sample_features)


Testing file: 0397.c02eba1386b00d640c954e5117dd1aa0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
====== INTEGRATED EMAIL FORENSIC REPORT ======

--- Parsed Email Details ---
From: sunday68@bluemail.dk
To: paulifree@hotmail.com
Subject: FORTUNE 500 WORK AT HOME REPS NEEDED!
Date: Fri, 20 Sep 2002 04:47:45 +0100
Return-Path: <sunday68@bluemail.dk>

--- Header Analysis Findings ---
- Return-Path found
- 4 Received headers detected
- No SPF/DKIM/DMARC authentication results present

--- ML Detection Results ---
Kaggle: Phishing | Score: 0.5032728314399719
SpamAssassin: Legitimate | Score: 0.4946371018886566
Trap4Phish: Legitimate | Score: 0.49498695135116577

--- Final Verdict ---
Suspicious / Potential Phishing

Testing file: 0259.7ebf3c0fd752bce5b8056e9454d2c76f
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
====== INTEGRATED EMAIL FORENSIC REPORT =====

In [ ]:
trap_df.head(1).T

,0
file_name,benign_sample_3253.xlsx
entropy_of_text,5.746068
macro_chr_count,39
macro_vocab_size,10079
macro_arithmetic_operator_count,4759
macro_token_count,10079
macro_max_line_length,38
remote_template_present,1
numeric_cell_count,4336
string_cell_count,8926


In [ ]:
ham_folder = "Datasets/spamassassin/20021010_easy_ham/easy_ham"

import os
print("Total legitimate emails:", len(os.listdir(ham_folder)))

Total legitimate emails: 2551


In [ ]:
ham_files = os.listdir(ham_folder)[:5]

for file in ham_files:
    print("\n==============================")
    print("Testing file:", file)

    email_path = os.path.join(ham_folder, file)

    with open(email_path, "r", encoding="latin-1") as f:
        raw_email = f.read()

    generate_integrated_report(raw_email, sample_feature)


Testing file: 1916.de9bd1f5764cf1615cfdfa0d7ecffeb1


NameError: name 'sample_feature' is not defined

In [ ]:
ham_files = os.listdir(ham_folder)[:5]

legit_count = 0
phish_count = 0

for file in ham_files:
    email_path = os.path.join(ham_folder, file)

    with open(email_path, "r", encoding="latin-1") as f:
        raw_email = f.read()

    result = generate_integrated_report(raw_email, sample_features)

    if result["final_verdict"] == "Legitimate":
        legit_count += 1
    else:
        phish_count += 1

print("Legitimate:", legit_count)
print("Suspicious:", phish_count)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
====== INTEGRATED EMAIL FORENSIC REPORT ======

Parsed Email Details:
{'from': 'gkm@petting-zoo.net', 'to': '0xdeadbeef@petting-zoo.net', 'subject': "New $199 PC Doesn't Do Windows", 'date': 'Wed, 02 Oct 2002 19:06:58 -0700', 'return_path': '<0xdeadbeef-request@petting-zoo.net>', 'received_headers': ['from localhost (jalapeno [127.0.0.1])\tby jmason.org (Postfix) with ESMTP id 3592216F03\tfor <jm@localhost>; Thu,  3 Oct 2002 12:22:47 +0100 (IST)', 'from jalapeno [127.0.0.1]\tby localhost with IMAP (fetchmail-5.9.0)\tfor jm@localhost (single-drop); Thu, 03 Oct 2002 12:22:47 +0100 (IST)', 'from petting-zoo.net (IDENT:postfix@petting-zoo.net    [64.166.12.219]) by dogma.slashnull.org (8.11.6/8.11.6) with ESMTP id    g9326sK10458 for <jm-deadbeef@jmason.org>; Thu, 3 Oct 2002 03:06:54 +0100', 'by petting-zoo.net (Postfix, from userid 1004) id 42441EA2E;    Wed,  2 Oct 2002 19:07

In [ ]:
result = generate_integrated_report(raw_email, sample_features)

type(result)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
====== INTEGRATED EMAIL FORENSIC REPORT ======

Parsed Email Details:
{'from': 'gkm@petting-zoo.net', 'to': '0xdeadbeef@petting-zoo.net', 'subject': "New $199 PC Doesn't Do Windows", 'date': 'Wed, 02 Oct 2002 19:06:58 -0700', 'return_path': '<0xdeadbeef-request@petting-zoo.net>', 'received_headers': ['from localhost (jalapeno [127.0.0.1])\tby jmason.org (Postfix) with ESMTP id 3592216F03\tfor <jm@localhost>; Thu,  3 Oct 2002 12:22:47 +0100 (IST)', 'from jalapeno [127.0.0.1]\tby localhost with IMAP (fetchmail-5.9.0)\tfor jm@localhost (single-drop); Thu, 03 Oct 2002 12:22:47 +0100 (IST)', 'from petting-zoo.net (IDENT:postfix@petting-zoo.net    [64.166.12.219]) by dogma.slashnull.org (8.11.6/8.11.6) with ESMTP id    g9326sK10458 for <jm-deadbeef@jmason.org>; Thu, 3 Oct 2002 03:06:54 +0100', 'by petting-zoo.net (Postfix, from userid 1004) id 42441EA2E;    Wed,  2 Oct 2002 19:07

dict

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [ ]:
test_results = []

def get_integrated_label(raw_email):
    result = generate_integrated_report(raw_email, sample_features)
    return result["final_verdict"]

# 10 spam emails
for file in os.listdir(spam_folder)[:10]:
    email_path = os.path.join(spam_folder, file)
    with open(email_path, "r", encoding="latin-1") as f:
        raw_email = f.read()

    pred = get_integrated_label(raw_email)
    test_results.append(["spam", pred])

# 10 legitimate emails
for file in os.listdir(ham_folder)[:10]:
    email_path = os.path.join(ham_folder, file)
    with open(email_path, "r", encoding="latin-1") as f:
        raw_email = f.read()

    pred = get_integrated_label(raw_email)
    test_results.append(["legitimate", pred])

test_results

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
====== INTEGRATED EMAIL FORENSIC REPORT ======

Parsed Email Details:
{'from': 'sunday68@bluemail.dk', 'to': 'paulifree@hotmail.com', 'subject': 'FORTUNE 500 WORK AT HOME REPS NEEDED!', 'date': 'Fri, 20 Sep 2002 04:47:45 +0100', 'return_path': '<sunday68@bluemail.dk>', 'received_headers': ['from localhost (jalapeno [127.0.0.1])\tby zzzzason.org (Postfix) with ESMTP id 8FD3716F03\tfor <zzzz@localhost>; Fri, 20 Sep 2002 11:41:21 +0100 (IST)', 'from jalapeno [127.0.0.1]\tby localhost with IMAP (fetchmail-5.9.0)\tfor zzzz@localhost (single-drop); Fri, 20 Sep 2002 11:41:21 +0100 (IST)', 'from webnote.net (mail.webnote.net [193.120.211.219]) by    dogma.slashnull.org (8.11.6/8.11.6) with ESMTP id g8K3lHC19280 for    <zzzz@jmason.org>; Fri, 20 Sep 2002 04:47:17 +0100', 'from bluemail.dk (host230-95.pool21756.interbusiness.it    [217.56.95.230]) by webnote.net (8.9.3/8.9.3) with SM

[['spam', 'Likely Legitimate'],
 ['spam', 'Likely Legitimate'],
 ['spam', 'Likely Legitimate'],
 ['spam', 'Likely Legitimate'],
 ['spam', 'Likely Legitimate'],
 ['spam', 'Likely Legitimate'],
 ['spam', 'Likely Legitimate'],
 ['spam', 'Likely Legitimate'],
 ['spam', 'Likely Legitimate'],
 ['spam', 'Suspicious / Potential Phishing'],
 ['legitimate', 'Suspicious / Potential Phishing'],
 ['legitimate', 'Likely Legitimate'],
 ['legitimate', 'Likely Legitimate'],
 ['legitimate', 'Likely Legitimate'],
 ['legitimate', 'Likely Legitimate'],
 ['legitimate', 'Likely Legitimate'],
 ['legitimate', 'Likely Legitimate'],
 ['legitimate', 'Likely Legitimate'],
 ['legitimate', 'Likely Legitimate'],
 ['legitimate', 'Likely Legitimate']]

In [ ]:
model_test_results = []

for file in os.listdir(spam_folder)[:10]:
    path = os.path.join(spam_folder, file)
    with open(path, "r", encoding="latin-1") as f:
        raw_email = f.read()
    body = clean_email_body(raw_email)

    kag = predict_kaggle_email(body)["prediction"]
    spam = predict_spamassassin_email(body)["prediction"]

    model_test_results.append(["spam", kag, spam])

for file in os.listdir(ham_folder)[:10]:
    path = os.path.join(ham_folder, file)
    with open(path, "r", encoding="latin-1") as f:
        raw_email = f.read()
    body = clean_email_body(raw_email)

    kag = predict_kaggle_email(body)["prediction"]
    spam = predict_spamassassin_email(body)["prediction"]

    model_test_results.append(["legitimate", kag, spam])

model_test_results

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━

[['spam', 'Phishing', 'Legitimate'],
 ['spam', 'Legitimate', 'Legitimate'],
 ['spam', 'Phishing', 'Legitimate'],
 ['spam', 'Phishing', 'Legitimate'],
 ['spam', 'Phishing', 'Legitimate'],
 ['spam', 'Phishing', 'Legitimate'],
 ['spam', 'Legitimate', 'Legitimate'],
 ['spam', 'Phishing', 'Legitimate'],
 ['spam', 'Legitimate', 'Spam/Phishing'],
 ['spam', 'Phishing', 'Spam/Phishing'],
 ['legitimate', 'Phishing', 'Spam/Phishing'],
 ['legitimate', 'Phishing', 'Legitimate'],
 ['legitimate', 'Phishing', 'Legitimate'],
 ['legitimate', 'Phishing', 'Legitimate'],
 ['legitimate', 'Phishing', 'Legitimate'],
 ['legitimate', 'Legitimate', 'Spam/Phishing'],
 ['legitimate', 'Phishing', 'Legitimate'],
 ['legitimate', 'Phishing', 'Legitimate'],
 ['legitimate', 'Phishing', 'Legitimate'],
 ['legitimate', 'Legitimate', 'Legitimate']]

In [ ]:
import pandas as pd
results_df = pd.DataFrame(
    model_test_results,
    columns=["Actual", "Kaggle", "SpamAssassin"]
)
results_df.head()

,Actual,Kaggle,SpamAssassin
0,spam,Phishing,Legitimate
1,spam,Legitimate,Legitimate
2,spam,Phishing,Legitimate
3,spam,Phishing,Legitimate
4,spam,Phishing,Legitimate


In [ ]:
results_df

,Actual,Kaggle,SpamAssassin
0,spam,Phishing,Legitimate
1,spam,Legitimate,Legitimate
2,spam,Phishing,Legitimate
3,spam,Phishing,Legitimate
4,spam,Phishing,Legitimate
5,spam,Phishing,Legitimate
6,spam,Legitimate,Legitimate
7,spam,Phishing,Legitimate
8,spam,Legitimate,Spam/Phishing
9,spam,Phishing,Spam/Phishing


In [ ]:
from sklearn.metrics import accuracy_score

kaggle_correct = (
    ((results_df["Actual"] == "spam") & (results_df["Kaggle"] == "Phishing")) |
    ((results_df["Actual"] == "legitimate") & (results_df["Kaggle"] == "Legitimate"))
)

spam_correct = (
    ((results_df["Actual"] == "spam") & (results_df["SpamAssassin"] == "Spam/Phishing")) |
    ((results_df["Actual"] == "legitimate") & (results_df["SpamAssassin"] == "Legitimate"))
)

print("Kaggle Accuracy:", kaggle_correct.mean())
print("SpamAssassin Accuracy:", spam_correct.mean())

Kaggle Accuracy: 0.45
SpamAssassin Accuracy: 0.5


In [ ]:
len(results_df)

20

In [ ]:
results = []

spam_files = os.listdir(spam_folder)[:100]
ham_files = os.listdir(ham_folder)[:100]

for file in spam_files:
    email_path = os.path.join(spam_folder, file)

    with open(email_path, "r", encoding="latin-1") as f:
        raw_email = f.read()

    result = generate_integrated_report(raw_email, sample_features)

    results.append(["spam", result["final_verdict"]])

for file in ham_files:
    email_path = os.path.join(ham_folder, file)

    with open(email_path, "r", encoding="latin-1") as f:
        raw_email = f.read()

    result = generate_integrated_report(raw_email, sample_features)

    results.append(["legitimate", result["final_verdict"]])

results_df = pd.DataFrame(results, columns=["Actual", "Prediction"])

results_df.head()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
====== INTEGRATED EMAIL FORENSIC REPORT ======

Parsed Email Details:
{'from': 'sunday68@bluemail.dk', 'to': 'paulifree@hotmail.com', 'subject': 'FORTUNE 500 WORK AT HOME REPS NEEDED!', 'date': 'Fri, 20 Sep 2002 04:47:45 +0100', 'return_path': '<sunday68@bluemail.dk>', 'received_headers': ['from localhost (jalapeno [127.0.0.1])\tby zzzzason.org (Postfix) with ESMTP id 8FD3716F03\tfor <zzzz@localhost>; Fri, 20 Sep 2002 11:41:21 +0100 (IST)', 'from jalapeno [127.0.0.1]\tby localhost with IMAP (fetchmail-5.9.0)\tfor zzzz@localhost (single-drop); Fri, 20 Sep 2002 11:41:21 +0100 (IST)', 'from webnote.net (mail.webnote.net [193.120.211.219]) by    dogma.slashnull.org (8.11.6/8.11.6) with ESMTP id g8K3lHC19280 for    <zzzz@jmason.org>; Fri, 20 Sep 2002 04:47:17 +0100', 'from bluemail.dk (host230-95.pool21756.interbusiness.it    [217.56.95.230]) by webnote.net (8.9.3/8.9.3) with SM

,Actual,Prediction
0,spam,Likely Legitimate
1,spam,Likely Legitimate
2,spam,Likely Legitimate
3,spam,Likely Legitimate
4,spam,Likely Legitimate


In [ ]:
kaggle_results = []

for file in os.listdir(spam_folder)[:100]:
    path = os.path.join(spam_folder, file)
    with open(path, "r", encoding="latin-1") as f:
        raw_email = f.read()

    body = clean_email_body(raw_email)
    pred = predict_kaggle_email(body)["prediction"]
    kaggle_results.append(["spam", pred])

for file in os.listdir(ham_folder)[:100]:
    path = os.path.join(ham_folder, file)
    with open(path, "r", encoding="latin-1") as f:
        raw_email = f.read()

    body = clean_email_body(raw_email)
    pred = predict_kaggle_email(body)["prediction"]
    kaggle_results.append(["legitimate", pred])

kaggle_df = pd.DataFrame(kaggle_results, columns=["Actual", "Prediction"])
kaggle_df.head()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━

,Actual,Prediction
0,spam,Phishing
1,spam,Legitimate
2,spam,Phishing
3,spam,Phishing
4,spam,Phishing


In [ ]:
kaggle_df["Prediction"].value_counts()

Prediction
Phishing      167
Legitimate     33
Name: count, dtype: int64

In [ ]:
pd.crosstab(kaggle_df["Actual"], kaggle_df["Prediction"])

Prediction,Legitimate,Phishing
Actual,,
legitimate,13,87
spam,20,80


In [ ]:
from sklearn.metrics import classification_report

y_true = ["Phishing" if x=="spam" else "Legitimate"
          for x in kaggle_df["Actual"]]

print(classification_report(
    y_true,
    kaggle_df["Prediction"]
))

              precision    recall  f1-score   support

  Legitimate       0.39      0.13      0.20       100
    Phishing       0.48      0.80      0.60       100

    accuracy                           0.47       200
   macro avg       0.44      0.47      0.40       200
weighted avg       0.44      0.47      0.40       200



In [75]:
generate_integrated_report(raw_email, sample_features)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
====== INTEGRATED EMAIL FORENSIC REPORT ======

Parsed Email Details:
{'from': 'Answer.Us@davicom.co.kr', 'to': 'undisclosed-recipients:;', 'subject': '$10 a hour for watching e-mmercials! No joke!', 'date': 'Mon, 07 Oct 2002 22:57:45 -0400', 'return_path': '<bounce2@u-answer.com>', 'received_headers': ['from localhost (jalapeno [127.0.0.1])\tby zzzzason.org (Postfix) with ESMTP id 10C9616F18\tfor <zzzz@localhost>; Tue,  8 Oct 2002 11:02:05 +0100 (IST)', 'from jalapeno [127.0.0.1]\tby localhost with IMAP (fetchmail-5.9.0)\tfor zzzz@localhost (single-drop); Tue, 08 Oct 2002 11:02:05 +0100 (IST)', 'from webnote.net (mail.webnote.net [193.120.211.219]) by    dogma.slashnull.org (8.11.6/8.11.6) with ESMTP id g98339K30569 for    <zzzz@jmason.org>; Tue, 8 Oct 2002 04:03:10 +0100', 'from davicom.co.kr (IDENT:root@[211.205.76.5]) by webnote.net    (8.9.3/8.9.3) with ESMTP id EAA211

{'email_text': 'Answer-Us Unlist Information This message is brought to you by Answer-us.com in \n\t\tcompliance with current federal laws. To find out more about Answer-us.com visit http://www.answer-us.com . \n        You are receiving this mailing because you or someone you know has \n        registered this email address to receive special offers from an Answer-us.com \n        marketing partner. Screening of addresses has been done to the best of \n        our knowledge. We honor all unlist requests within 72 hours. If you\n        have received this email in error, we apologize for any inconvenience it \n        has caused and will not mail further offers to you. To be\n        unlisted from our database, please do the \n        following: Simply click here . If you have your mail forwarded to a new email address \n        please provide your old email address. A nswer - Us Nationwide support@answer-us.com Our \n        E-Mail Campaigns Have Produced\n      Staggering Response Ra